load_ext autoreload: Prevents you from having to restart the kernel.

autoreload 2: Sets the reload mode to "Reload everything".

ANYWIDGET_HMR=1: Configures Anywidget to listen for changes in the backend.

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


# Importing a Barchart written in D3.js

Based on the notebook [Barchart](https://observablehq.com/@d3/bar-chart/2) by Observable.


The notebook shows us the following D3.js code

```js
chart = {
  // Declare the chart dimensions and margins.
  const width = 928;
  const height = 500;
  const marginTop = 30;
  const marginRight = 0;
  const marginBottom = 30;
  const marginLeft = 40;

  // Declare the x (horizontal position) scale.
  const x = d3.scaleBand()
      .domain(d3.groupSort(data, ([d]) => -d.frequency, (d) => d.letter)) // descending frequency
      .range([marginLeft, width - marginRight])
      .padding(0.1);
  
  // Declare the y (vertical position) scale.
  const y = d3.scaleLinear()
      .domain([0, d3.max(data, (d) => d.frequency)])
      .range([height - marginBottom, marginTop]);

  // Create the SVG container.
  const svg = d3.create("svg")
      .attr("width", width)
      .attr("height", height)
      .attr("viewBox", [0, 0, width, height])
      .attr("style", "max-width: 100%; height: auto;");

  // Add a rect for each bar.
  svg.append("g")
      .attr("fill", "steelblue")
    .selectAll()
    .data(data)
    .join("rect")
      .attr("x", (d) => x(d.letter))
      .attr("y", (d) => y(d.frequency))
      .attr("height", (d) => y(0) - y(d.frequency))
      .attr("width", x.bandwidth());

  // Add the x-axis and label.
  svg.append("g")
      .attr("transform", `translate(0,${height - marginBottom})`)
      .call(d3.axisBottom(x).tickSizeOuter(0));

  // Add the y-axis and label, and remove the domain line.
  svg.append("g")
      .attr("transform", `translate(${marginLeft},0)`)
      .call(d3.axisLeft(y).tickFormat((y) => (y * 100).toFixed()))
      .call(g => g.select(".domain").remove())
      .call(g => g.append("text")
          .attr("x", -marginLeft)
          .attr("y", 10)
          .attr("fill", "currentColor")
          .attr("text-anchor", "start")
          .text("↑ Frequency (%)"));

  // Return the SVG element.
  return svg.node();
}
```

This generates a graph like the following:

![image](https://raw.githubusercontent.com/MatiasMaravi/D3Bridge/main/D3Bridge/images/chart.png)

A test dataset titled *alphabet* is also used, which contains the frequency with which letters of the alphabet are repeated in a given text.

For more details on how to use D3Bridge datasets, see the notebook [Datasets](dataset.ipynb).

In [2]:
import D3Bridge.datasets as ds

df_alphabet = ds.get_dataset("alphabet")
df_alphabet.columns


['letter', 'frequency']

In [3]:
from D3Bridge import CustomWidget
import traitlets

In [ ]:
class CustomBarplot(CustomWidget):
    #_esm = CustomWidget.createWidgetFromLocalFile(paramList=["data"], filePath=r"d3_barchart_original.js")
    _esm = CustomWidget.createWidgetFromUrl(paramList=["data"], jsUrl=r"https://raw.githubusercontent.com/MatiasMaravi/D3Bridge/main/D3Bridge/samples/d3_barchart_original.js")
    data = traitlets.List([]).tag(sync=True)

barplot = CustomBarplot(data=df_alphabet.data)

In [5]:
barplot

This was made possible because we modified some important lines in the original code.

```js
function plot(data) { //Main line changed, we need to create the "plot" class and specify the parameters.

  const marginTop = 30;
  const marginRight = 0;
  const marginBottom = 30;
  const marginLeft = 40;


  d3.select(element).selectAll("*").remove(); //Second main line, it is necessary to rerender when there is a change in the data.
  const x = d3.scaleBand()
      .domain(d3.groupSort(data, ([d]) => -d.frequency, (d) => d.letter))
      .range([marginLeft, width - marginRight])
      .padding(0.1);
  

  const y = d3.scaleLinear()
      .domain([0, d3.max(data, (d) => d.frequency)])
      .range([height - marginBottom, marginTop]);


  const svg = d3.select(element) // Last main line; it is necessary to add the svg to the main element of the widget.
      .append("svg") // Inside the svg is the graphic, and inside element are all the Custom Widget attributes necessary to display it in a cell.
      .attr("width", width)
      .attr("height", height)
      .attr("viewBox", [0, 0, width, height])
      .attr("style", "max-width: 100%; height: auto;");


  svg.append("g")
      .attr("fill", "steelblue")
    .selectAll()
    .data(data)
    .join("rect")
      .attr("x", (d) => x(d.letter))
      .attr("y", (d) => y(d.frequency))
      .attr("height", (d) => y(0) - y(d.frequency))
      .attr("width", x.bandwidth());


  svg.append("g")
      .attr("transform", `translate(0,${height - marginBottom})`)
      .call(d3.axisBottom(x).tickSizeOuter(0));


  svg.append("g")
      .attr("transform", `translate(${marginLeft},0)`)
      .call(d3.axisLeft(y).tickFormat((y) => (y * 100).toFixed()))
      .call(g => g.select(".domain").remove())
      .call(g => g.append("text")
          .attr("x", -marginLeft)
          .attr("y", 10)
          .attr("fill", "currentColor")
          .attr("text-anchor", "start")
          .text("↑ Frequency (%)"));

}
```

If you want your chart to be reusable for other datasets, you need to specify more elements. In the case of the Barchart, you need to indicate the names of the columns that will be our X-axis and our Y-axis.

In [ ]:
df_explorers = ds.read_dataset(r"https://gist.githubusercontent.com/MatiasMaravi/5e540b21eb04ca980d77199317e56ebb/raw/Explorers.tsv",sep="\t")
class CustomBarplot_2(CustomWidget):
    #_esm = CustomWidget.createWidgetFromLocalFile(paramList=["data","x_","y_","pallete"], filePath=r"d3_barchart_modified.js")
    _esm = CustomWidget.createWidgetFromUrl(paramList=["data","x_","y_","pallete"], jsUrl=r"https://raw.githubusercontent.com/MatiasMaravi/D3Bridge/main/D3Bridge/samples/d3_barchart_modified.js")
    
    data = traitlets.List([]).tag(sync=True)
    x_ = traitlets.Unicode("").tag(sync=True)
    y_ = traitlets.Unicode("").tag(sync=True)
    pallete = traitlets.List([]).tag(sync=True)

barplot2 = CustomBarplot_2(data=df_explorers.data, x_="explorer", y_="frequency", pallete=["#1f77b4"])

In [7]:
barplot2

```js
function plot(data,x_,y_,pallete) { //New parameters defined
  //The variable "letter" was changed to "x_" and the variable "frequency" was changed to "y_".
  //The ability to change the bar chart color by passing a list of color palettes was also added.
  const marginTop = 30;
  const marginRight = 20;
  const marginBottom = 30;
  const marginLeft = 40;

  d3.select(element).selectAll("*").remove();

  const x = d3.scaleBand()
      .domain(d3.groupSort(data, ([d]) => -d[y_], (d) => d[x_]))
      .range([marginLeft, width - marginRight])
      .padding(0.1);
  
  const y = d3.scaleLinear()
      .domain([0, d3.max(data, (d) => d[y_])])
      .range([height - marginBottom, marginTop]);


  const svg = d3.select(element)
      .append("svg")
      .attr("width", width)
      .attr("height", height)
      .attr("viewBox", [0, 0, width, height])
      .attr("style", "max-width: 100%; height: auto;");


  svg.append("g")
    .selectAll()
    .data(data)
    .join("rect")
      .attr("x", (d) => x(d[x_]))
      .attr("y", (d) => y(d[y_]))
      .attr("height", (d) => y(0) - y(d[y_]))
      .attr("width", x.bandwidth())
      .attr("fill", (d, i) => pallete[i % pallete.length]); //Line added to change the color of the bars

  svg.append("g")
      .attr("transform", `translate(0,${height - marginBottom})`)
      .call(d3.axisBottom(x).tickSizeOuter(0));


  svg.append("g")
      .attr("transform", `translate(${marginLeft},0)`)
      .call(d3.axisLeft(y).tickFormat((y) => (y * 100).toFixed()))
      .call(g => g.select(".domain").remove())
      .call(g => g.append("text")
          .attr("x", -marginLeft)
          .attr("y", 10)
          .attr("fill", "currentColor")
          .attr("text-anchor", "start")
          .text("↑ Frequency (%)"));
}
```

To see an example of how to use the color palette provided by our library, check the notebook [colors](colors.ipynb)